# `ptof_obs_hallucination_detection`

## What this notebook does
Answers "did the agent tell the truth about what it was given" -- the core of the agent
input/output relationship. Combines three independent signal layers into one risk verdict per
call: **Layer 1** (an LLM-judge `verify_grounding` capability's verdict, currently always
NULL/empty -- see the open gap below), **Layer 2** (`ai_similarity` between response and prompt,
scored incrementally via a watermark), and **Layer 3** (regex-based ungrounded-token detection --
numbers/codes in the response that don't appear in the prompt).

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `04_hallucination_detection` -- runs in parallel with
  `02_latency_detection`, `03_malformed_output`, `05_behavioral_correlation`, after
  `01_bronze_projections`, before `06_alert`.
- **Upstream:** reads `v_llm_bronze` (built by `ptof_obs_bronze_projection`),
  `capability_registry` (human-curated by `ptof_obs_setup_seed`, joined as
  `r.active = true AND r.is_generative = true` -- non-generative capabilities are structurally
  exempt from grounding checks), `_obs_watermark` (the incremental-scoring bookmark, seeded by
  `ptof_obs_setup_seed`).
- **Downstream:** `ptof_obs_alert.ipynb` reads `hallucination_signal` (v1 detector
  `hallucination_high`, CRITICAL) -- only 2 scored rows exist system-wide as of this writing,
  both `low` risk: armed and correct, but data-starved. Its value grows with call volume.

## Known open gap (Phase 4, item 9 of the implementation plan)
Layer 1 joins against a `verify_grounding` capability that doesn't exist in this data, so
`hallucination_verdicts` is always empty and `vg_verdict` is always NULL for every row. This is a
documented, known-expected state (see `ptof_obs_alert.ipynb`'s header), not a bug -- Layers 2 and 3
carry the actual signal today. Not yet formally labeled `NOT YET ACTIVE` in `threshold_basis` per
the implementation plan's Phase 4 remediation item.

## Tables/views touched
- **Reads:** `v_llm_bronze`, `capability_registry`, `_obs_watermark`,
  `hallucination_verdicts`/`faithfulness_scores`/`v_ungrounded_tokens` (this notebook's own
  outputs, joined together in the final cell).
- **Writes:** `hallucination_verdicts` (Layer 1, currently always empty),
  `faithfulness_scores` (Layer 2, incrementally MERGEd), `v_ungrounded_tokens` (Layer 3, a VIEW
  not a table -- always current), `hallucination_signal` (the combined verdict `ptof_obs_alert`
  reads), and `_obs_watermark` (advanced only after Layer 2 scoring succeeds).


In [0]:
%sql
-- hallucination_verdicts — Layer 1: an LLM-judge verdict on whether a response was grounded in
-- its prompt. Currently ALWAYS EMPTY: the WHERE clause filters to capability = 'verify_grounding',
-- which doesn't exist in this data (see this notebook's overview above and
-- ptof_obs_alert.ipynb's "Known-expected-states" note). Layers 2 and 3 below carry the real
-- signal today; this stays wired so it activates automatically if verify_grounding ever ships.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.hallucination_verdicts AS
WITH ordered AS (
  -- Pairs each verify_grounding call with the call immediately preceding it in the same
  -- shift/batch -- the LAG() lookup that would let a judge's verdict attach to the response it
  -- actually judged, once such calls exist.
  SELECT
      id, shift_date, shift_type, batch_nbr, capability, success,
      called_at, response_parsed, resp_v, user_prompt,
      LAG(id)              OVER w AS prior_id,
      LAG(capability)      OVER w AS prior_capability,
      LAG(response_parsed) OVER w AS prior_response_parsed,
      LAG(user_prompt)     OVER w AS prior_user_prompt,
      LAG(called_at)       OVER w AS prior_called_at
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze
  WINDOW w AS (
    PARTITION BY shift_date, shift_type, batch_nbr
    ORDER BY called_at
  )
)
SELECT
    prior_id       AS verified_row_id,
    id             AS verify_grounding_row_id,
    shift_date, shift_type, batch_nbr,
    prior_capability       AS verified_capability,
    prior_response_parsed  AS verified_response,
    prior_user_prompt      AS grounding_input,
    resp_v:verdict::string    AS vg_verdict,
    resp_v:confidence::double AS vg_confidence,
    unix_timestamp(called_at) - unix_timestamp(prior_called_at) AS vg_gap_s,
    called_at AS vg_called_at
FROM ordered
WHERE capability = 'verify_grounding'
  AND prior_id IS NOT NULL
  AND prior_capability <> 'verify_grounding'      -- ← the new line
  AND unix_timestamp(called_at) - unix_timestamp(prior_called_at) BETWEEN 0 AND 900;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- faithfulness_scores — Layer 2: ai_similarity(response, prompt) as a numeric grounding proxy,
-- scored incrementally past the watermark rather than rescanning all of v_llm_bronze every run
-- (the one place in this codebase using the watermark+MERGE incremental pattern, seeded by
-- ptof_obs_setup_seed's _obs_watermark table).
-- Layer 2. Scores only rows past the watermark; without this, ai_similarity is never invoked
-- and every new row lands in hallucination_signal as 'unverified'.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.faithfulness_scores (
    id STRING, shift_date STRING, shift_type STRING, batch_nbr STRING,
    capability STRING, called_at TIMESTAMP,
    resp_vs_prompt_similarity DOUBLE,
    similarity_pctile_in_capability DOUBLE
);

MERGE INTO mq_gmdf_dev.oil_obs.faithfulness_scores t
USING (
  SELECT b.id, b.shift_date, b.shift_type, b.batch_nbr, b.capability, b.called_at,
         ai_similarity(cast(b.response_parsed AS STRING),
                       cast(b.user_prompt     AS STRING)) AS resp_vs_prompt_similarity
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true AND r.is_generative = true
  WHERE b.called_at > (SELECT last_processed_ts FROM mq_gmdf_dev.oil_obs._obs_watermark
                        WHERE detector = 'faithfulness_scores')
    AND b.called_at <= current_timestamp()
    AND b.success = true
    AND b.is_blank_output        = false
    AND b.is_credential_fastfail = false
) s
ON t.id = s.id
WHEN NOT MATCHED THEN INSERT (id, shift_date, shift_type, batch_nbr, capability, called_at,
                              resp_vs_prompt_similarity)
  VALUES (s.id, s.shift_date, s.shift_type, s.batch_nbr, s.capability, s.called_at,
          s.resp_vs_prompt_similarity);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
-- v_ungrounded_tokens -- Layer 3: regex-based detection of specific numbers/codes that
-- appear in the response but not the prompt (a cheap, judge-free proxy for "the agent
-- invented a number"). A VIEW, not a table, so it's always current -- no incremental/
-- watermark bookkeeping needed since regex extraction is cheap relative to Layer 2's
-- ai_similarity calls.
-- v_ungrounded_tokens
-- Regex additions: [A-Z]{1,4}\d{3,} catches E007896, FILL3320, TT3990_REF3275A, QBMS3990 —
-- all of which the previous \b-anchored numeric pattern missed entirely.
-- Normalization: strips % and thousands separators, trims trailing .0, so prompt -205200 and
-- response -205,200 no longer read as ungrounded (confirmed false positive on dsa_compare).
CREATE OR REPLACE VIEW mq_gmdf_dev.oil_obs.v_ungrounded_tokens AS
WITH raw AS (
  SELECT
      id, shift_date, shift_type, batch_nbr, capability, called_at,
      regexp_extract_all(cast(response_parsed AS STRING),
        '(?i)\\b([A-Z]{1,4}\\d{3,}(?:_[A-Za-z0-9]+)*|\\d{1,7}(?:[.,]\\d+)?%?|\\d{1,2}:\\d{2}|\\d{4}-\\d{2}-\\d{2})\\b', 1) AS resp_raw,
      regexp_extract_all(cast(user_prompt AS STRING),
        '(?i)\\b([A-Z]{1,4}\\d{3,}(?:_[A-Za-z0-9]+)*|\\d{1,7}(?:[.,]\\d+)?%?|\\d{1,2}:\\d{2}|\\d{4}-\\d{2}-\\d{2})\\b', 1) AS prompt_raw
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze
  WHERE success = true
    AND is_blank_output        = false
    AND is_credential_fastfail = false
),
norm AS (
  -- strip formatting differences (%, thousands separators, trailing .0) before comparing,
  -- so the same number formatted two ways in prompt vs. response isn't a false-positive
  -- mismatch.
  SELECT id, shift_date, shift_type, batch_nbr, capability, called_at,
         array_distinct(transform(resp_raw,   x ->
           regexp_replace(regexp_replace(x, '[%,]', ''), '\\.0+$', ''))) AS resp_tokens,
         array_distinct(transform(prompt_raw, x ->
           regexp_replace(regexp_replace(x, '[%,]', ''), '\\.0+$', ''))) AS prompt_tokens
  FROM raw
)
-- ungrounded_tokens: tokens present in the response but absent from the prompt -- what
-- hallucination_signal's ungrounded_token_count branch reads.
SELECT id, shift_date, shift_type, batch_nbr, capability, called_at,
       array_except(resp_tokens, prompt_tokens)       AS ungrounded_tokens,
       size(array_except(resp_tokens, prompt_tokens)) AS ungrounded_token_count
FROM norm
WHERE size(array_except(resp_tokens, prompt_tokens)) > 0;

In [0]:
%sql
-- NOTE: this cell duplicates cell 2 above (identical MERGE into faithfulness_scores). Running it
-- twice per pass is harmless -- the second MERGE's USING subquery re-reads the same
-- past-the-watermark window and every row from the first pass already matches on t.id, so
-- WHEN NOT MATCHED never fires again -- but it is redundant compute, not a second distinct step.
-- Layer 2: ai_similarity scoring, incremental past the watermark.
-- Without this cell ai_similarity is never invoked and every new row lands in
-- hallucination_signal as 'unverified'. That is what took unverified from 1-of-7 to 12-of-14.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.faithfulness_scores (
    id STRING, shift_date STRING, shift_type STRING, batch_nbr STRING,
    capability STRING, called_at TIMESTAMP,
    resp_vs_prompt_similarity DOUBLE,
    similarity_pctile_in_capability DOUBLE
);

MERGE INTO mq_gmdf_dev.oil_obs.faithfulness_scores t
USING (
  SELECT b.id, b.shift_date, b.shift_type, b.batch_nbr, b.capability, b.called_at,
         ai_similarity(cast(b.response_parsed AS STRING),
                       cast(b.user_prompt     AS STRING)) AS resp_vs_prompt_similarity
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true AND r.is_generative = true
  WHERE b.called_at > (SELECT last_processed_ts FROM mq_gmdf_dev.oil_obs._obs_watermark
                        WHERE detector = 'faithfulness_scores')
    AND b.called_at <= current_timestamp()
    AND b.success = true
    AND b.is_blank_output        = false
    AND b.is_credential_fastfail = false
) s
ON t.id = s.id
WHEN NOT MATCHED THEN INSERT (id, shift_date, shift_type, batch_nbr, capability, called_at,
                              resp_vs_prompt_similarity)
  VALUES (s.id, s.shift_date, s.shift_type, s.batch_nbr, s.capability, s.called_at,
          s.resp_vs_prompt_similarity);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
-- Recomputes each row's similarity percentile WITHIN its own capability -- the number
-- hallucination_signal actually thresholds on (a fixed similarity cutoff would treat a
-- naturally-more-varied capability the same as a naturally-consistent one).
-- Percentiles are relative, so they shift as rows are added. MERGE rather than
-- CREATE OR REPLACE ... AS SELECT ... FROM itself, which drops and recreates the table.
MERGE INTO mq_gmdf_dev.oil_obs.faithfulness_scores t
USING (
  SELECT id,
         percent_rank() OVER (PARTITION BY capability ORDER BY resp_vs_prompt_similarity)
           AS pctile
  FROM mq_gmdf_dev.oil_obs.faithfulness_scores
) s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.similarity_pctile_in_capability = s.pctile;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
193,193,0,0


In [0]:
%sql
-- Advances the incremental-scoring bookmark so the next run only scores rows newer than this
-- one. Kept as its own cell deliberately:
-- Separate cell on purpose: if scoring above fails, the watermark must NOT advance, so the next
-- run re-scans the same range.
UPDATE mq_gmdf_dev.oil_obs._obs_watermark
SET last_processed_ts = current_timestamp(), updated_at = current_timestamp()
WHERE detector = 'faithfulness_scores';

num_affected_rows
1


In [0]:
%sql
-- hallucination_signal — combines all three layers into one hallucination_risk verdict per
-- call. This is what ptof_obs_alert's hallucination_high detector (CRITICAL in v1) reads.
-- hallucination_signal — driven FROM v_llm_bronze, not from the empty hallucination_verdicts.
-- Layers 2 and 3 now work standalone, which matters because verify_grounding does not exist.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.hallucination_signal AS
SELECT
    b.id                        AS row_id,
    b.shift_date, b.shift_type, b.batch_nbr,
    b.capability                AS verified_capability,
    b.called_at,                                  -- the time anchor the alert needs
    v.vg_verdict, v.vg_confidence, v.vg_called_at, v.vg_gap_s,
    f.resp_vs_prompt_similarity,
    f.similarity_pctile_in_capability,
    d.ungrounded_token_count,
    d.ungrounded_tokens,
    -- hallucination_risk: the combined verdict. v.vg_verdict is always NULL today (Layer 1's
    -- known gap), so in practice this resolves via Layers 2/3 -- the percentile+similarity-floor
    -- branch, or the ungrounded-token-count branch for GxP-relevant capabilities.
    CASE
      WHEN v.vg_verdict = 'ungrounded' AND v.vg_confidence >= 0.7        THEN 'high'
      WHEN v.vg_verdict = 'ungrounded'
       AND (f.similarity_pctile_in_capability < 0.10
            OR d.ungrounded_token_count > 3)                             THEN 'high'
      WHEN d.ungrounded_token_count > 3 AND r.is_gxp_relevant = true      THEN 'high'
            -- Absolute floor alongside the percentile: percent_rank() over a single-row capability is
      -- always 0, so dsa_optimize (n=1, similarity 0.887 — the highest in the data) would score
      -- medium purely by construction. Observed band is 0.69-0.89; 0.75 is provisional pending
      -- Phase 3 calibration.
      WHEN v.vg_verdict IS NULL
       AND f.similarity_pctile_in_capability < 0.05
       AND f.resp_vs_prompt_similarity < 0.75                            THEN 'medium'
      WHEN v.vg_verdict IS NULL
       AND f.resp_vs_prompt_similarity IS NULL                           THEN 'unverified'
      ELSE 'low'
    END AS hallucination_risk,
    current_timestamp() AS detected_at
FROM      mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN      mq_gmdf_dev.oil_obs.capability_registry r
       ON r.capability = b.capability AND r.active = true AND r.is_generative = true
LEFT JOIN mq_gmdf_dev.oil_obs.hallucination_verdicts v ON v.verified_row_id = b.id
LEFT JOIN mq_gmdf_dev.oil_obs.faithfulness_scores    f ON f.id = b.id
LEFT JOIN mq_gmdf_dev.oil_obs.v_ungrounded_tokens    d ON d.id = b.id
WHERE b.success = true
  AND b.is_blank_output        = false
  AND b.is_credential_fastfail = false
  AND b.called_at >= current_timestamp() - INTERVAL 24 HOURS;

num_affected_rows,num_inserted_rows
